# 10. Ekstraksi Fitur Gabungan (204 Fitur) — Pendekatan Linier & Polynomial

Notebook ini menghasilkan dua file CSV berisi **204 fitur gabungan** dari tiga polutan
(**NO2**, **CO**, **SO2**) yang diekstraksi menggunakan 68 fungsi TSFEL.
Perbedaan antara dua file terletak pada **metode imputasi missing value** yang dipakai:

| File Keluaran | Metode Imputasi | Keterangan |
|---|---|---|
| `fitur_all_linear_bundah_sreseh.csv` | Interpolasi **Linier** (time-based) | Metode yang digunakan pada Tugas 5 & 6 |
| `fitur_all_polynomial_bundah_sreseh.csv` | Interpolasi **Polynomial** (orde 2, kuadratik) | Pendekatan alternatif yang menangkap kurva lengkung antartitik |

Adapun **preprocessing** yang diterapkan sebelum ekstraksi fitur sama untuk keduanya:
deteksi dan penghapusan outlier dengan metode **IQR**, diikuti imputasi
missing value dengan metode yang berbeda, kemudian ekstraksi 68 fitur TSFEL.

> **Format output:** 1 baris × 204 kolom. Setiap kolom diberi awalan nama polutan,
> misalnya `NO2_abs_energy`, `CO_spectral_entropy`, `SO2_kurtosis`, dst.


## 1. Persiapan Pustaka dan Konfigurasi


In [1]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features
import warnings
warnings.filterwarnings('ignore')

# Pemetaan nama kolom data (lowercase) ke label prefix fitur (uppercase)
POLLUTANTS = {
    'no2': 'NO2',
    'co':  'CO',
    'so2': 'SO2',
}

# Daftar 68 fitur TSFEL yang identik dengan notebook sebelumnya
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

fs = 1  # 1 sampel per hari

print(f"Jumlah polutan  : {len(POLLUTANTS)}")
print(f"Jumlah fitur/polutan: {len(FEATURE_LIST)}")
print(f"Total kolom output  : {len(POLLUTANTS) * len(FEATURE_LIST)}")


Jumlah polutan      : 3
Jumlah fitur/polutan: 68
Total kolom output  : 204


## 2. Fungsi Bantu

Dua fungsi disiapkan:
- `to_scalar()`: memastikan hasil TSFEL yang berupa list/array dikembalikan sebagai
  satu nilai numerik (rata-rata), kompatibel dengan fitur multi-nilai seperti `mfcc`, `lpcc`, `ecdf_percentile`.
- `extract_one()`: memanggil satu fungsi TSFEL berdasarkan namanya secara dinamis,
  mendeteksi otomatis apakah fungsi tersebut memerlukan parameter `fs` atau tidak.


In [ ]:
def to_scalar(result):
    """Konversi hasil TSFEL (scalar/list/dict) ke satu nilai float."""
    if isinstance(result, dict) and 'values' in result:
        result = result['values']
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    """Panggil fungsi TSFEL berdasarkan nama string, dengan/tanpa parameter fs."""
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    result = fn(signal, fs) if 'fs' in params else fn(signal)
    return to_scalar(result)


def clean_and_extract(df_raw, col, prefix, interp_method='linear', poly_order=2):
    """
    Preprocessing + ekstraksi 68 fitur TSFEL untuk satu polutan.

    Parameters
    ----------
    df_raw        : DataFrame dengan kolom 'date' dan 'col'
    col           : nama kolom polutan (str)
    prefix        : awalan nama fitur keluaran, mis. 'NO2'
    interp_method : 'time' untuk linier berbasis waktu,
                    'polynomial' untuk kuadratik (perlu poly_order)
    poly_order    : orde polynomial (digunakan saat method='polynomial')

    Returns
    -------
    dict {prefix_fnname: nilai}
    """
    df = df_raw[['date', col]].copy()
    df[col] = pd.to_numeric(df[col], errors='coerce')

    n_before = df[col].isna().sum()

    # Hapus outlier dengan IQR
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lb, ub = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_outlier = ((df[col] < lb) | (df[col] > ub)).sum()
    df.loc[(df[col] < lb) | (df[col] > ub), col] = np.nan

    n_total_nan = df[col].isna().sum()

    # Imputasi sesuai metode yang dipilih
    df_idx = df.set_index('date')
    if interp_method == 'polynomial':
        df_idx[col] = df_idx[col].interpolate(method='polynomial', order=poly_order)
    else:
        df_idx[col] = df_idx[col].interpolate(method='time')
    df_idx[col] = df_idx[col].ffill().bfill()

    n_after = df_idx[col].isna().sum()
    signal = df_idx[col].astype(float).values

    print(f"  {prefix}: NaN awal={n_before}, outlier IQR={n_outlier}, "
          f"total NaN sebelum imputasi={n_total_nan}, sisa setelah imputasi={n_after}")
    print(f"  IQR bounds: [{lb:.6f}, {ub:.6f}]")

    # Ekstraksi 68 fitur
    row = {}
    for fn_name in FEATURE_LIST:
        row[f"{prefix}_{fn_name}"] = extract_one(fn_name, signal, fs)
    return row


## 3. Memuat Dataset

Dataset utama `kualitas_udara_bundah_sreseh.csv` sudah memuat ketiga polutan dalam satu file.
Kolom `date` dikonversi ke format `datetime` agar interpolasi berbasis waktu berjalan benar.


In [2]:
df_main = pd.read_csv('../data/kualitas_udara_bundah_sreseh.csv')
df_main['date'] = pd.to_datetime(df_main['date'])
df_main = df_main.sort_values('date').reset_index(drop=True)

print(f"Dimensi dataset : {df_main.shape}")
print(f"Rentang tanggal : {df_main['date'].min().date()} s.d. {df_main['date'].max().date()}")
print(f"Kolom           : {df_main.columns.tolist()}")
print()
for col in ['no2', 'co', 'so2']:
    print(f"  {col.upper()}: {df_main[col].notna().sum()} data valid, "
          f"{df_main[col].isna().sum()} NaN dari {len(df_main)} total hari")


Dataset: (362, 4)  |  2025-09-01 - 2026-08-29
Kolom           : ['date', 'no2', 'co', 'so2']

  NO2: 190 data valid, 172 NaN dari 362 total hari
  CO : 172 data valid, 190 NaN dari 362 total hari
  SO2: 238 data valid, 124 NaN dari 362 total hari


## 4. Ekstraksi Fitur — Imputasi Linier (Time-based)

Pada pendekatan **linier**, nilai NaN diisi dengan interpolasi berbasis waktu
(`method='time'`) yang mengasumsikan perubahan nilai berjalan secara **linier dan proporsional**
antara dua titik data valid yang mengapitnya.
Ini adalah metode yang sama dengan yang digunakan pada Tugas 5 dan 6.


In [3]:
print('=== IMPUTASI LINIER ===\n')
combined_linear = {}
for col, prefix in POLLUTANTS.items():
    print(f'--- {prefix} ---')
    row = clean_and_extract(df_main, col, prefix, interp_method='time')
    combined_linear.update(row)
    print()

df_linear = pd.DataFrame([combined_linear])
print(f'Total kolom: {df_linear.shape[1]}')
df_linear.head(1)


=== IMPUTASI LINIER ===

--- NO2 ---
  NO2: NaN awal=172, outlier IQR=3, total NaN sblm imputasi=175, sisa setelah imputasi=0
  IQR bounds: [-0.000008, 0.000050]

--- CO ---
  CO: NaN awal=190, outlier IQR=1, total NaN sblm imputasi=191, sisa setelah imputasi=0
  IQR bounds: [0.018035, 0.038864]

--- SO2 ---
  SO2: NaN awal=124, outlier IQR=11, total NaN sblm imputasi=135, sisa setelah imputasi=0
  IQR bounds: [-0.000251, 0.000302]

Total kolom: 204


In [4]:
output_linear = '../data/fitur_all_linear_bundah_sreseh.csv'
df_linear.to_csv(output_linear, index=False)
print(f'File disimpan: {output_linear}')
print(f'Dimensi      : {df_linear.shape}  (baris × kolom)')


File disimpan: ../data/fitur_all_linear_bundah_sreseh.csv
Dimensi      : (1, 204)  (baris x kolom)


## 5. Ekstraksi Fitur — Imputasi Polynomial (Kuadratik, Orde 2)

Pada pendekatan **polynomial**, nilai NaN diisi dengan interpolasi kuadratik (`order=2`)
yang menyesuaikan kurva orde-2 melalui titik-titik data di sekitar celah kosong.
Pendekatan ini lebih fleksibel dari interpolasi linier:

- Lebih akurat di segmen data yang tren-nya **melengkung** (percepatan/perlambatan polutan musiman).
- Bisa menghasilkan nilai sedikit berbeda di luar rentang yang linier tidak mampu menangkap.

Kedua file akhir kemudian dapat digunakan sebagai bahan perbandingan dalam analisis
apakah metode imputasi berpengaruh terhadap kualitas fitur dan hasil clustering.


In [5]:
print('=== IMPUTASI POLYNOMIAL (orde 2) ===\n')
combined_poly = {}
for col, prefix in POLLUTANTS.items():
    print(f'--- {prefix} ---')
    row = clean_and_extract(df_main, col, prefix, interp_method='polynomial', poly_order=2)
    combined_poly.update(row)
    print()

df_poly = pd.DataFrame([combined_poly])
print(f'Total kolom: {df_poly.shape[1]}')
df_poly.head(1)


=== IMPUTASI POLYNOMIAL (orde 2) ===

--- NO2 ---
  NO2: NaN awal=172, outlier IQR=3, total NaN sblm imputasi=175, sisa setelah imputasi=0
  IQR bounds: [-0.000008, 0.000050]

--- CO ---
  CO: NaN awal=190, outlier IQR=1, total NaN sblm imputasi=191, sisa setelah imputasi=0
  IQR bounds: [0.018035, 0.038864]

--- SO2 ---
  SO2: NaN awal=124, outlier IQR=11, total NaN sblm imputasi=135, sisa setelah imputasi=0
  IQR bounds: [-0.000251, 0.000302]

Total kolom: 204


In [6]:
output_poly = '../data/fitur_all_polynomial_bundah_sreseh.csv'
df_poly.to_csv(output_poly, index=False)
print(f'File disimpan: {output_poly}')
print(f'Dimensi      : {df_poly.shape}  (baris × kolom)')


File disimpan: ../data/fitur_all_polynomial_bundah_sreseh.csv
Dimensi      : (1, 204)  (baris x kolom)


## 6. Perbandingan Distribusi Fitur: Linear vs Polynomial

Untuk memahami seberapa besar pengaruh metode interpolasi terhadap nilai fitur yang dihasilkan,
kita bandingkan distribusi **fitur yang sama** antara dua metode interpolasi.

Visualisasi berikut menampilkan nilai setiap fitur (diurutkan dari kecil ke besar berdasarkan
nilai Linear) untuk ketiga polutan secara terpisah. Titik **lingkaran** (●) mewakili nilai dari
interpolasi **Linier**, sedangkan titik **berlian** (◆) mewakili nilai dari interpolasi **Polynomial**.

Karena data kita berasal dari satu lokasi pengamatan (1 baris × 204 fitur), setiap "posisi" pada
sumbu X merepresentasikan satu fitur TSFEL yang sudah diurutkan — bukan urutan wilayah.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams['figure.dpi'] = 110

# Muat kedua file hasil ekstraksi
df_lin = pd.read_csv('../data/fitur_all_linear_bundah_sreseh.csv')
df_pol = pd.read_csv('../data/fitur_all_polynomial_bundah_sreseh.csv')

PREFIXES = {
    'NO2': {'color_lin': '#2563EB', 'color_pol': '#60A5FA', 'label': 'no2'},
    'SO2': {'color_lin': '#BE185D', 'color_pol': '#F472B6', 'label': 'so2'},
    'CO':  {'color_lin': '#15803D', 'color_pol': '#4ADE80', 'label': 'co'},
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Perbandingan Fitur: Interpolasi Linear vs Polynomial', fontsize=13, fontweight='bold')

for ax, (prefix, cfg) in zip(axes, PREFIXES.items()):
    # Ambil kolom untuk polutan ini
    cols = [c for c in df_lin.columns if c.startswith(f'{prefix}_')]
    lin_vals = df_lin[cols].iloc[0].values.astype(float)
    pol_vals = df_pol[cols].iloc[0].values.astype(float)

    # Normalisasi ke skala z-score agar bisa dibandingkan dalam satu plot
    mu, sigma = np.nanmean(lin_vals), np.nanstd(lin_vals)
    if sigma == 0:
        sigma = 1
    lin_z = (lin_vals - mu) / sigma
    pol_z = (pol_vals - mu) / sigma

    # Urutkan berdasarkan nilai linear (ascending)
    sort_idx = np.argsort(lin_z)
    x = np.arange(len(sort_idx))

    ax.scatter(x, lin_z[sort_idx], s=30, marker='o',
               color=cfg['color_lin'], label='Linear', zorder=3)
    ax.scatter(x, pol_z[sort_idx], s=30, marker='D',
               color=cfg['color_pol'], label='Polynomial', alpha=0.8, zorder=2)

    ax.set_title(f'{prefix}_{cfg["label"]}_all_features\n(diurutkan per nilai fitur)', fontsize=10)
    ax.set_xlabel('Urutan Fitur (setelah sort)', fontsize=9)
    ax.set_ylabel('Nilai fitur (z-score)', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Tabel ringkasan perbedaan
diff = (df_lin - df_pol).abs()
top_diff = pd.DataFrame({
    'Linear': df_lin.iloc[0],
    'Polynomial': df_pol.iloc[0],
    'AbsDiff': diff.iloc[0],
}).sort_values('AbsDiff', ascending=False)

print('\nFitur dengan perbedaan terbesar antara Linear dan Polynomial:')
print(top_diff.head(10).to_string())
print(f'\nFitur identik (diff=0) : {(diff.iloc[0] == 0).sum()}')
print(f'Fitur berbeda          : {(diff.iloc[0] != 0).sum()}')


## 7. Ringkasan

| | Linier | Polynomial (orde 2) |
|---|---|---|
| **Metode imputasi** | `interpolate(method='time')` | `interpolate(method='polynomial', order=2)` |
| **Jumlah kolom output** | 204 (68 × 3 polutan) | 204 (68 × 3 polutan) |
| **Outlier handling** | IQR (sama) | IQR (sama) |
| **File output** | `fitur_all_linear_bundah_sreseh.csv` | `fitur_all_polynomial_bundah_sreseh.csv` |
| **Asumsi gap data** | Perubahan linier/lurus | Perubahan mengikuti kurva kuadratik |

Kedua file siap digunakan dalam tahap clustering lanjutan sebagai input
204 fitur gabungan tiga polutan.
